In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [2]:
df = pd.read_excel(
    r'E:\pycharm all files\眼动数据处理\GSR\完整SCL处理分析\文件提取&GSR分解&去除异常值&绘图\GSR_SCL_SCR_AllEvents.xlsx')
df

,组别,姓名,飞行天数,阶段,data
0,A,付瑞晗,2,hejiu,1.561250e+06
1,A,付瑞晗,2,hejiu,1.561945e+06
2,A,付瑞晗,2,hejiu,1.561511e+06
3,A,付瑞晗,2,hejiu,1.561072e+06
4,A,付瑞晗,2,hejiu,1.561473e+06
...,...,...,...,...,...
405101,B,陈妍,7,jiangluo,9.336832e+05
405102,B,陈妍,7,jiangluo,9.337367e+05
405103,B,陈妍,7,jiangluo,9.339863e+05
405104,B,陈妍,7,jiangluo,9.343174e+05


In [10]:
df.阶段.unique()

array(['hejiu', 'qifei', '1', '2', '3', '4', 'jiangluo'], dtype=object)

In [8]:
# 数据时长,4HZ
print('数据时长：', len(df) / 4, '秒')

数据时长： 101276.5 秒


In [13]:
# 各阶段均值标准差
# 计算每条记录的时长（秒）
# data列应该是某种信号值，长度就是记录数，所以直接按阶段统计行数
# 如果每行就是一个采样点，则：
sampling_interval = 0.25  # 秒

# 先把阶段 1,2,3,4 合并为“转弯”
df['阶段'] = df['阶段'].replace(['1', '2', '3', '4'], '转弯')
df['阶段'] = df['阶段'].replace('hejiu', '静息')
df['阶段'] = df['阶段'].replace('qifei', '起飞')
df['阶段'] = df['阶段'].replace('jiangluo', '降落')
# 按组别、姓名、飞行天数、阶段统计每组的记录数

# 按组别、姓名、飞行天数、阶段统计每次的记录时长
duration_df = (
    df.groupby(['组别', '姓名', '飞行天数', '阶段'])
    .size()  # 每组有多少行
    .reset_index(name='采样点数')
)
duration_df['记录时长_秒'] = duration_df['采样点数'] * sampling_interval

# 再按阶段统计均值和标准差
result = (
    duration_df.groupby('阶段')['记录时长_秒']
    .agg(['mean', 'std'])
    .reset_index()
)

print('各阶段均值标准差：')

# 打印均值 ± 标准差
for _, row in result.iterrows():
    print(f"阶段 {row['阶段']} : {row['mean']:.2f} ± {row['std']:.2f} 秒")


各阶段均值标准差：
阶段 起飞 : 64.69 ± 22.66 秒
阶段 转弯 : 95.74 ± 21.53 秒
阶段 降落 : 67.11 ± 35.41 秒
阶段 静息 : 299.94 ± 62.47 秒


In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# 读取数据
df = pd.read_excel(
    r'E:\pycharm all files\眼动数据处理\GSR\完整SCL处理分析\文件提取&GSR分解&去除异常值&绘图\GSR_SCL_SCR_AllEvents.xlsx'
)

# 阶段合并
df['阶段'] = df['阶段'].replace(
    {'1': '转弯', '2': '转弯', '3': '转弯', '4': '转弯',
     'hejiu': '静息', 'qifei': '起飞', 'jiangluo': '降落'}
)

# 每个阶段记录时长（秒）
sampling_interval = 0.25  # 4Hz
duration_df = (
    df.groupby(['组别', '姓名', '飞行天数', '阶段'])
    .size()
    .reset_index(name='采样点数')
)
duration_df['记录时长_秒'] = duration_df['采样点数'] * sampling_interval

# 计算每个被试一天总时长
daily_duration = (
    duration_df.groupby(['组别', '姓名', '飞行天数'])['记录时长_秒']
    .sum()
    .reset_index(name='一天总时长_秒')
)

# 全体均值±标准差
mean_daily = daily_duration['一天总时长_秒'].mean()
std_daily = daily_duration['一天总时长_秒'].std()

print(f"被试一天实验总时长（全体）：{mean_daily:.2f} ± {std_daily:.2f} 秒")


被试一天实验总时长（全体）：527.48 ± 77.29 秒
